# ING Employerer-Branding Diagnosis: ING vs Santander, BNP-Paribas & Deutsche-Bank

**Dataset:** Glassdoor Job Reviews 1 & 2 — David Gauthier ([Kaggle](https://www.kaggle.com/datasets/davidgauthier/glassdoor-job-reviews)). Merged and filtered into a single Parquet file (July 2018 - July 2023, firms with 300+ reviews).

**Objective:** Identify ING's strengths (to exploit in employer branding and recruitment) and improvement areas (the ones driving employees out), always compared against its main competitors.

## 0. SetUp

### 0.1. Imports

In [15]:
# Standard library
import re
from pathlib import Path

# Dataframe management
import pandas as pd
import numpy as np

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

### 0.2. Project Config

In [16]:
# Data_Path
DATA_PATH = Path("data/data.parquet")

# Selected firms
TARGET = "ING"
PEERS = ["Santander", "BNP-Paribas", "Deutsche-Bank"]
FIRMS = [TARGET] + PEERS

# Reviews CAP per firm ~ 1500
MAX_REVIEWS_PER_FIRM = 1500

# Reproducibility
SEED = 5

# Plot style ~ Color Branding
sns.set_theme(style = "whitegrid")

PALETTE = {
    TARGET: "#FF6200",
    "Santander": "#EC0000",
    "BNP-Paribas": "#00915A",
    "Deutsche-Bank": "#0018A8"
    }

### 0.3. Data import

In [17]:
# RELEVANT data import from data/data.parquet
full_df = pd.read_parquet(DATA_PATH, filters=[("firm", "in", FIRMS)])

### 0.4. Balanced Sampling

The whole project (EDA and NLP) works with a balanced sample of `MAX_REVIEWS_PER_FIRM` reviews per firm, so no firm has more weight than the others in the joint topic model and computing time stays reasonable for the transformer models. Sampling happens right after loading, so I don't clean or explore reviews that won't be used.

To keep each firm's own composition, the sample is stratified by year and current/former status (proportional allocation), with a fixed seed so the whole project is reproducible.

In [19]:
# Stratification by Year & Current/Former
strata = pd.to_datetime(full_df["date_review"]).dt.year.astype(str) + "_" + full_df["current"].str.split().str[0]

samples = []

for firm, g in full_df.groupby("firm"):

    # % Reviews to keep
    fraction = min(1, MAX_REVIEWS_PER_FIRM / len(g))

    # Stratified sampling
    out = g.groupby(strata[g.index]).sample(frac=fraction, random_state=SEED)

    # Rounding in each stratum can overshoot the cap, so trim back to MAX_REVIEWS_PER_FIRM.
    if len(out) > MAX_REVIEWS_PER_FIRM:
        out = out.sample(n = MAX_REVIEWS_PER_FIRM, random_state=SEED)

    samples.append(out)

# Sampled DF (working df)
raw_df = pd.concat(samples, ignore_index=True)

print(f"Working sample: {len(raw_df)} reviews")

Working sample: 5998 reviews


**Sampling check**

Composition of the sample vs. the full data. `mean_rating` was not used to stratify, so it is a real test of representativeness.

In [24]:
def composition(df):
    return pd.DataFrame({
        "pct_current": df.groupby("firm")["current"].apply(lambda s: s.str.lower().str.startswith("current").mean() * 100),
        "mean_rating": df.groupby("firm")["overall_rating"].mean(),
    })

sampling_check = (pd.DataFrame({"available": full_df["firm"].value_counts(), "sample": raw_df["firm"].value_counts()})
                  .join(composition(full_df).add_suffix("_full"))
                  .join(composition(raw_df).add_suffix("_sample"))
                  .loc[FIRMS].round(2))

sampling_check

,available,sample,pct_current_full,mean_rating_full,pct_current_sample,mean_rating_sample
firm,,,,,,
ING,1681,1500,67.28,4.00,67.27,4.00
Santander,2638,1499,60.65,3.57,60.71,3.59
BNP-Paribas,4455,1499,64.62,3.66,64.64,3.68
Deutsche-Bank,5474,1500,62.99,3.73,63.00,3.71


## 1. EDA

### 1.1. DataFrame Quality Analysis

**DataFrame dimensions**

In [20]:
print(f"Rows: {raw_df.shape[0]}")
print(f"Cols: {raw_df.shape[1]}")

Rows: 5998
Cols: 7


**Col type (& nulls)**

In [21]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5998 entries, 0 to 5997
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   firm            5998 non-null   str    
 1   date_review     5998 non-null   object 
 2   current         5998 non-null   str    
 3   overall_rating  5998 non-null   float64
 4   recommend       5998 non-null   str    
 5   pros            5998 non-null   str    
 6   cons            5998 non-null   str    
dtypes: float64(1), object(1), str(5)
memory usage: 1.3+ MB


**Duplicates Count**

In [22]:
dupl_count = raw_df.duplicated().sum()
print("Duplicates: ", dupl_count)

Duplicates:  0


**Reviews per firm**

In [23]:
print(raw_df["firm"].value_counts())

firm
Deutsche-Bank    1500
ING              1500
BNP-Paribas      1499
Santander        1499
Name: count, dtype: int64


### 1.2. Col Casting & Parsing

**Fixing Current**

`Current` mixes employment status and tenure (e.g. _"Former Employee, more than 3 years"_), so I split it into two variables.

In [26]:
# Current / Former
def get_status(s):
    s = str(s).lower()
    if s.startswith("current"):
        return "Current"
    if s.startswith("former"):
        return "Former"
    return "Unknown"

# Tenure
def get_tenure(s):
    m = re.search(r"(less|more) than (\d+) year", str(s))
    return f"{m.group(1)} than {m.group(2)}y" if m else "n/a"

# Applying
raw_df["status"] = raw_df["current"].map(get_status)
raw_df["tenure"] = raw_df["current"].map(get_tenure)

**Recomemend Fixing**

`Recommend` is translated from its code (v = yes, o = no opinion, x = no).

In [27]:
raw_df["recommends"] = raw_df["recommend"].map({"v": "Yes", "o": "Neutral", "x": "No"})

**Date casting**

In [28]:
# Casting
raw_df["date_review"] = pd.to_datetime(raw_df["date_review"])

# Extracting year
raw_df["year"] = raw_df["date_review"].dt.year

**Categorical firm**

This way ING will always be the first firm.

In [29]:
raw_df["firm"] = pd.Categorical(raw_df["firm"], categories=FIRMS, ordered=True)

**Text cleaning**

In [30]:
# Normalizing whitespaces
for col in ["pros", "cons"]:
    raw_df[col] = raw_df[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

### 1.3. Data Understanding

Before comparing firms, I check how each variable is distributed to know what the data looks like and which analyses are viable. The whole EDA uses the balanced sample (`clean_df`).

In [31]:
clean_df = raw_df.copy()

**Date Range**

In [32]:
print("\nDate range:", raw_df["date_review"].min(), "->", raw_df["date_review"].max())


Date range: 2018-07-27 00:00:00 -> 2023-04-23 00:00:00


**Reviews per year and firm**

# VAS POR AQUÍ ANIMAL

In [33]:
# Reviews per year and firm. 2018 and 2023 are partial years (July 2018 - July 2023)
print(clean_df.pivot_table(index="year", columns="firm", values="overall_rating", aggfunc="count", observed=True)[FIRMS])

firm  ING  Santander  BNP-Paribas  Deutsche-Bank
year                                            
2018   31         45           53             83
2019  150        172          190            205
2020  193        292          215            237
2021  506        540          476            449
2022  463        341          462            425
2023  157        109          103            101


**Rating distribution**

In [ ]:
# % of reviews by score, for each firm
rating_dist = pd.crosstab(clean_df["firm"], clean_df["overall_rating"], normalize="index").mul(100)

fig, ax = plt.subplots(figsize=(10, 5))
rating_dist.T.plot(kind="bar", color=[PALETTE[firm] for firm in rating_dist.index], ax=ax)

ax.set_title("Rating distribution by firm")
ax.set_xlabel("Rating")
ax.set_ylabel("% of reviews")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Firm")

plt.tight_layout()
plt.show()

> **TODO:** Is the rating concentrated in the extremes? If so, the mean alone can be misleading and I'll complement it with the % of high and low scores.

**Recommendation, status & tenure**

In [ ]:
# Share of each category (% of reviews)
for col in ["recommends", "status", "tenure"]:
    print(clean_df[col].value_counts(normalize=True, dropna=False).mul(100).round(1), "\n")

> **TODO:** Is `Neutral` a relevant share of `recommends`? Does `n/a` dominate `tenure`? If it does, a tenure analysis is not viable.

**Text length**

In [ ]:
clean_df["n_words_pros"] = clean_df["pros"].str.split().str.len()
clean_df["n_words_cons"] = clean_df["cons"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ["n_words_pros", "n_words_cons"], ["Pros", "Cons"]):
    sns.boxplot(data=clean_df, x="firm", y=col, order=FIRMS, showfliers=False, showmeans=True,
                meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black"}, ax=ax)
    ax.set_title(f"{title}: words per review (diamond = mean)")
    ax.set_ylabel("Words")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Transformers truncate at 512 tokens (~350-400 words), so check how many reviews would be cut
for col in ["n_words_pros", "n_words_cons"]:
    print(f"{col}: {round((clean_df[col] > 350).mean() * 100, 2)}% of reviews above 350 words")

> **TODO:** If the share of reviews above 350 words is small, truncation is not a real problem for the sentiment model and I'll keep `truncation=True` as is.

### 1.4. Business focused analysis

**Note:** Numbers above bars represent the number of reviews, not the rating.

In [ ]:
# Auxiliar func. to plot mean rating with review counts
def barplot_with_counts(data, x, order, ax, title):
    sns.barplot(data=data, x=x, y="overall_rating", order=order, ax=ax, errorbar=None, color="#9E9E9E")
    counts = data[x].value_counts()
    for i, category in enumerate(order):
        n = counts.get(category, 0)
        height = data[data[x] == category]["overall_rating"].mean()
        ax.text(i, height + 0.05, f"{n}", ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_ylim(0, 5)
    ax.set_ylabel("Mean rating (1-5)")

**Firm summary**

In [ ]:
# Recommendation rate is computed only over reviews with an opinion (Yes / No). "Neutral" = no opinion
with_opinion = clean_df[clean_df["recommends"].isin(["Yes", "No"])]

summary = pd.DataFrame({
    "reviews": clean_df["firm"].value_counts(),
    "mean_rating": clean_df.groupby("firm", observed=True)["overall_rating"].mean(),
    "pct_recommend": with_opinion.groupby("firm", observed=True)["recommends"].apply(lambda s: (s == "Yes").mean() * 100),
    "pct_current": clean_df.groupby("firm", observed=True)["status"].apply(lambda s: (s == "Current").mean() * 100),
}).loc[FIRMS].round(2)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean rating with 95% CI, so small differences between firms are not over-interpreted
sns.pointplot(data=clean_df, x="firm", y="overall_rating", order=FIRMS, errorbar=("ci", 95),
              linestyle="none", color="#424242", ax=axes[0])
axes[0].set_title("Mean rating by firm (95% CI)")
axes[0].set_ylabel("Mean rating (1-5)")
axes[0].tick_params(axis="x", rotation=30)

# Recommendation rate
sns.barplot(x=summary.index, y=summary["pct_recommend"], hue=summary.index, palette=PALETTE, order=FIRMS, legend=False, ax=axes[1])
axes[1].set_title("% of employees who recommend the firm")
axes[1].set_ylabel("% recommend")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

> **TODO:** Is ING above or below the peers' average in rating and recommendation? Do the confidence intervals overlap? If they do, the difference is not reliable and I shouldn't present it as a finding.

**Rating ~ Employment status (current vs. former)**

In [ ]:
status_df = clean_df[clean_df["status"] != "Unknown"]

gap = status_df.pivot_table(index="firm", columns="status", values="overall_rating", aggfunc="mean", observed=True).loc[FIRMS]
gap["gap (Current - Former)"] = gap["Current"] - gap["Former"]
print(gap.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=status_df, x="firm", y="overall_rating", hue="status", order=FIRMS,
            hue_order=["Current", "Former"], errorbar=None, ax=ax)
ax.set_title("Mean rating: current vs. former employees")
ax.set_ylabel("Mean rating (1-5)")
ax.set_ylim(0, 5)
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

Former employees are expected to rate lower in every firm (selection bias: people who leave are, on average, less satisfied), so the absolute difference is not informative by itself. What matters is whether ING's gap is larger than its peers'.

> **TODO:** Compare ING's gap against the peers' gaps.

**Rating evolution**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for firm in FIRMS:
    firm_year = clean_df[clean_df["firm"] == firm].groupby("year")["overall_rating"].mean()
    ax.plot(firm_year.index, firm_year.values, marker="o", color=PALETTE[firm],
            lw=3.5 if firm == TARGET else 1.5, label=firm)

ax.set_title("Mean rating by year")
ax.set_ylabel("Mean rating (1-5)")
ax.set_xticks(sorted(clean_df["year"].unique()))
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()
plt.show()

> **TODO:** Does ING improve or worsen over time relative to its peers? Is there any inflection point (e.g. 2020)? If a year has few reviews, I won't interpret it.

**Rating ~ Recommendation**

In [ ]:
# If rating and recommendation tell the same story, rating should drop from Yes -> Neutral -> No
rec_order = ["Yes", "Neutral", "No"]

fig, ax = plt.subplots(figsize=(7, 5))
barplot_with_counts(clean_df.dropna(subset=["recommends"]), "recommends", rec_order, ax, "Mean rating by recommendation")

plt.tight_layout()
plt.show()

print(clean_df.pivot_table(index="firm", columns="recommends", values="overall_rating", aggfunc="mean", observed=True)
      .loc[FIRMS][rec_order].round(2))

This is a consistency check rather than an insight: if `overall_rating` and `recommend` disagreed, I couldn't use either of them later to validate the topics.

> **TODO:** Confirm the expected pattern holds in all four firms.